# One-dimensional heat conduction with finite differences

Copyright © 2026 Philip Eisenlohr and contributors. [License](https://github.com/mseMSU/Notebooks-pub/blob/main/LICENSE.md)

In [ ]:
# Load the shared notebook utilities locally or, in Colab, from GitHub.
try:
    import nbkit
except ModuleNotFoundError:
    from pathlib import Path
    from urllib.request import urlretrieve
    import sys
    import tempfile

    module_path = next(
        (candidate
         for parent in (Path.cwd(), *Path.cwd().parents)
         for candidate in (parent / 'nbkit.py', parent / 'notebooks' / 'nbkit.py')
         if candidate.is_file()),
        None,
    )

    if module_path is None:
        support_dir = Path(tempfile.gettempdir()) / 'notebook_tools'
        support_dir.mkdir(exist_ok=True)
        module_path = support_dir / 'nbkit.py'
        urlretrieve(
            'https://raw.githubusercontent.com/'
            'mseMSU/Notebooks-pub/main/notebooks/nbkit.py',
            module_path,
        )

    sys.path.insert(0, str(module_path.parent))
    import nbkit


## Overview

This starter notebook illustrates how a finite-difference scheme turns a one-dimensional partial differential equation into repeated array updates.
We model a uniform rod whose ends are held at a reference temperature and watch an initially warm interior cool by heat conduction.
The example is intentionally minimal and can be expanded into a fuller lesson later.

After working through it, you should be able to identify a spatial grid, a time step, boundary conditions, and the stability limit of an explicit scheme.

## Approximating derivatives on a fixed grid

A computer cannot store a continuous field at every possible position and time.
We instead sample it on a fixed, uniformly spaced grid

$$
x_i=i\,\Delta x, \qquad t_n=n\,\Delta t, \qquad \theta_i^n\approx\theta(x_i,t_n).
$$

The subscripts identify positions and the superscripts identify time levels.

### First derivatives

A difference between two neighboring samples measures the average slope between them, so a first-order difference naturally belongs halfway between the samples.

For example, the spatial and temporal forward differences are

$$
\left.\frac{\partial\theta}{\partial x}\right|_{i+1/2}^{n}\approx\frac{\theta_{i+1}^{n}-\theta_i^{n}}{\Delta x}, \qquad
\left.\frac{\partial\theta}{\partial t}\right|_{i}^{n+1/2}\approx\frac{\theta_i^{n+1}-\theta_i^n}{\Delta t}.
$$

![Forward spatial and temporal differences located between stored grid points](assets/finite_difference_heat_equation/first_derivatives.svg)

The half-indices in the diagram are conceptual locations rather than additional stored grid points.
They record where the difference quotient is naturally centered.
The spatial quotient between $i$ and $i+1$, for example, is associated with $i+1/2$ even though the array stores values only at integer indices.

If the first spatial derivative is needed at the stored point $i$, we can average the slopes on its left and right:

$$
\left.\frac{\partial\theta}{\partial x}\right|_i^n
\approx\frac{1}{2}\left(
\frac{\theta_i^n-\theta_{i-1}^n}{\Delta x}
+\frac{\theta_{i+1}^n-\theta_i^n}{\Delta x}
\right)
=\frac{\theta_{i+1}^n-\theta_{i-1}^n}{2\Delta x}.
$$

This is the centered-difference approximation to the first spatial derivative.
A centered temporal derivative at $t_n$ similarly uses values at $t_{n-1}$ and $t_{n+1}$.
During explicit time marching, however, $t_{n+1}$ is the unknown level we are trying to calculate, so we instead interpret the temporal quotient between $n$ and $n+1$ as a forward-Euler approximation at $t_n$.

### Second derivatives

A second spatial derivative measures how the first spatial derivative changes.
Define the two neighboring slope approximations as

$$
q_{i-1/2}^n=\frac{\theta_i^n-\theta_{i-1}^n}{\Delta x}, \qquad
q_{i+1/2}^n=\frac{\theta_{i+1}^n-\theta_i^n}{\Delta x}.
$$

These first derivatives occupy the two conceptual half-points on either side of $i$.
Their difference is centered back at the original stored grid point $i$.

![A second spatial derivative formed from neighboring first derivatives](assets/finite_difference_heat_equation/second_derivative.svg)

Dividing the change in these slopes by the distance between their half-point locations gives

$$
\left.\frac{\partial^2\theta}{\partial x^2}\right|_i^n
\approx\frac{1}{\Delta x}
\left(
\frac{\theta_{i+1}^n-\theta_i^n}{\Delta x}
-\frac{\theta_i^n-\theta_{i-1}^n}{\Delta x}
\right)
=\frac{\theta_{i+1}^n-2\theta_i^n+\theta_{i-1}^n}{\Delta x^2}.
$$

### Grid refinement and accuracy

Difference quotients approach the corresponding derivatives as $\Delta x$ and $\Delta t$ approach zero, provided the underlying solution is sufficiently smooth.
The centered first and second spatial differences above have truncation errors proportional to $\Delta x^2$, whereas the forward-Euler time approximation has an error proportional to $\Delta t$.
Consequently, halving $\Delta x$ ideally reduces the spatial discretization error by roughly a factor of four, while halving $\Delta t$ reduces the temporal discretization error by roughly a factor of two.
A finer grid also requires more stored values and more calculations.
For the explicit heat-equation scheme used below, reducing $\Delta x$ additionally forces a reduction of $\Delta t$ to preserve stability, so spatial refinement can substantially increase the number of time steps.
Comparing results on successively refined grids is therefore an important practical test that the numerical solution is converging.

## Toy model: one-dimensional heat conduction

Let $\theta(x,t)=T(x,t)-T_{\mathrm{ref}}$ denote temperature excess above a reference temperature.
For a uniform rod with constant thermal diffusivity $\alpha$ and no internal heat generation, conservation of energy leads to the heat equation

$$
\frac{\partial \theta}{\partial t}=\alpha\frac{\partial^2\theta}{\partial x^2}, \qquad 0<x<L.
$$

Here, $L$ is measured in metres, time in seconds, $\alpha$ in square metres per second, and $\theta$ in kelvin.
The time derivative describes local change, while the second spatial derivative measures curvature and therefore how different a point is from its surroundings.

## Initial and boundary conditions

The differential equation alone does not determine a unique temperature history.
We must specify the temperature throughout the rod at the initial time and describe how both ends interact with their surroundings.

Common boundary-condition choices include:

- A **fixed value** or Dirichlet condition prescribes the temperature at an end, such as $\theta(0,t)=0$.
- A **fixed slope** or Neumann condition prescribes $\partial\theta/\partial x$ and therefore the conductive heat flux; a zero slope represents an insulated end.
- A **mixed** or Robin condition relates temperature to its slope and can model convective heat transfer to the surroundings.
- A **periodic** condition connects the two ends and is useful when the modeled interval repeats.

Dirichlet conditions are especially simple numerically because their boundary-node values can be assigned directly after every time step.
Slope and mixed conditions require an additional finite-difference approximation at the boundary, often using a one-sided difference or an auxiliary ghost point.

For this toy problem, both ends are held at the reference temperature:

$$
\theta(0,t)=\theta(L,t)=0.
$$

We choose an initially warm sinusoidal interior that already satisfies those end values:

$$
\theta(x,0)=A\sin(\pi x/L).
$$

Physically, this represents a rod placed in contact at both ends with ideal temperature reservoirs that remain unaffected by the heat they receive.

## Assemble the explicit solution scheme

We approximate the time derivative between levels $n$ and $n+1$ with a forward difference and evaluate the centered second spatial derivative using values from level $n$:

$$
\frac{\theta_i^{n+1}-\theta_i^n}{\Delta t}
=\alpha\frac{\theta_{i+1}^n-2\theta_i^n+\theta_{i-1}^n}{\Delta x^2}.
$$

Solving this algebraic equation for the only unknown, $\theta_i^{n+1}$, gives the explicit update

$$
\theta_i^{n+1}=\theta_i^n+r\left(\theta_{i+1}^n-2\theta_i^n+\theta_{i-1}^n\right), \qquad r=\frac{\alpha\Delta t}{\Delta x^2}.
$$

Every interior value at the new time level must be calculated from the same old time level.
The two boundary values are then reset to zero to enforce the chosen Dirichlet conditions.
For this constant-coefficient, one-dimensional explicit scheme on a uniform grid, stability requires $r\leq 1/2$; see [Chapter 20, *Finite difference schemes for the heat equation in one dimension*](https://userpages.umbc.edu/~rostamia/cbook/fd1/finite-differences.pdf) for a derivation.
A time step that violates this restriction can make rounding and discretization errors grow until the numerical result oscillates or diverges.

Before revealing or running the outputs, predict how the peak height and the shape of the temperature profile will change.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 0.1                         # Rod length, m
alpha = 1.0e-4                  # Thermal diffusivity, m^2/s
amplitude = 100.0               # Initial peak temperature excess, K
nx = 51                         # Grid points, including the two boundaries
t_final = 5.0                   # Final time, s
target_r = 0.4                  # Safely below the explicit stability limit

x = np.linspace(0.0, L, nx)
dx = x[1] - x[0]
nsteps = int(np.ceil(t_final / (target_r * dx**2 / alpha)))
dt = t_final / nsteps           # Adjust slightly to land exactly at t_final
r = alpha * dt / dx**2
assert 0.0 < r <= 0.5

theta = amplitude * np.sin(np.pi * x / L)
theta[[0, -1]] = 0.0
snapshots = [(0.0, theta.copy())]
save_steps = set(np.linspace(0, nsteps, 5, dtype=int)[1:])

for step in range(1, nsteps + 1):
    old = theta.copy()
    theta[1:-1] = old[1:-1] + r * (
        old[2:] - 2.0 * old[1:-1] + old[:-2]
    )
    theta[[0, -1]] = 0.0
    if step in save_steps:
        snapshots.append((step * dt, theta.copy()))

print(f"dx = {dx:.4f} m, dt = {dt:.5f} s, r = {r:.3f}")
print(f"Time steps: {nsteps}")
print(f"Final peak temperature excess: {theta.max():.2f} K")


## Compare with a known solution

For this particular initial profile and these boundary conditions, the exact solution is

$$
\theta(x,t)=A\sin(\pi x/L)\exp\left[-\alpha(\pi/L)^2t\right].
$$

Its shape stays sinusoidal while its amplitude decays.
We compare this expression with the numerical solution at the final time.
Agreement here provides a useful check, although it is not a proof that every possible input or implementation is correct.

In [ ]:
exact = amplitude * np.sin(np.pi * x / L) * np.exp(
    -alpha * (np.pi / L)**2 * t_final
)
error = np.max(np.abs(theta - exact))
print(f"Maximum absolute error at t = {t_final:g} s: {error:.4f} K")

fig, ax = plt.subplots()
colors = plt.cm.Blues(np.linspace(0.35, 0.95, len(snapshots)))
for (time, profile), color in zip(snapshots, colors):
    ax.plot(x, profile, color=color, linewidth=1.5,
            zorder=3, label=f"t = {time:.2f} s")
# Plot the exact solution last for its legend position but below the curves.
ax.plot(x, exact, color="black", linestyle=":", linewidth=4,
        zorder=2, label="Exact solution at final time")
ax.set_xlabel("Position / m")
ax.set_ylabel("Temperature excess / K")
ax.set_title("Cooling of a rod with fixed-temperature ends")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Try next

1. Increase the thermal diffusivity and predict how much faster the rod cools before running the notebook again.
2. Refine the grid by increasing `nx`, keeping `target_r` fixed, and compare the final error and number of time steps.
3. Replace the initial profile with another smooth shape that is zero at both ends.
   The displayed exact solution applies only to the original sine profile, so remove that comparison for other initial conditions.

Future extensions could introduce boundary heat fluxes, internal heat generation, implicit time stepping, and a systematic convergence study.
